# Homework 11: querying a materials database

Covers Lecture 29 (querying Materials Project in a loop), Lecture 30 (composition as a feature), Lecture 31 (stability and the convex hull), and Lecture 32 (missing values and provenance).

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning.
Reloading or disconnecting can revert the notebook to the instructor copy.

# Problem 0 (-10 pts if blank): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work. Critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

# Setup

Run these two cells first.
They need the `MaterialsProject` secret from the API-key checklist on Canvas.

> The install takes a minute or two. If Colab asks you to restart the session, restart and run both cells again.

In [ ]:
!pip install mp-api

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
from mp_api.client import MPRester

API_KEY = userdata.get('MaterialsProject')     # your key, read from Colab Secrets

# Problem 1 (30 pts): querying a database in a loop

The Period 4 transition metals run from scandium (Sc) to zinc (Zn).

(a) Make a list `metals` holding the ten chemical symbols from Sc to Zn, in order.

In [ ]:
# your code here

(b) Build a `DataFrame` called `metal_table` with the material ID and density of the stable crystal of each metal.

In [ ]:
# your code here

(c) Plot the density of each metal as a bar chart, in the order of `metals`.

In [ ]:
# your code here

(d) These are measured densities at room temperature (*CRC Handbook of Chemistry and Physics*):

| metal | measured density (g/cm$^3$) |
|---|---|
| Ti | 4.51 |
| Fe | 7.87 |
| Cu | 8.96 |

- Print the Materials Project density next to the measured one for each of the three.
- Describe what each density column represents.
- Give one reason they can differ.

In [ ]:
# your code here

*your answer here*

# Problem 2 (25 pts): composition as a feature

Iron and oxygen form several compounds (FeO, Fe$_3$O$_4$, Fe$_2$O$_3$, and others), each in one or more crystal structures.
Problems 2, 3, and 4(c) use every Fe-O entry in the database, stable or not.

(a) Query every Fe-O entry, stable or not.

- Print how many entries came back.

> Problems 2-4 use `formula_pretty`, `composition`, `formation_energy_per_atom`, `energy_above_hull`, `is_stable`, and `theoretical`.

In [ ]:
# your code here

(b) Build a `DataFrame` called `fe_o` with one row per entry, including the atomic fraction of oxygen as a column `x_O`.

In [ ]:
# your code here

(c) Plot formation energy per atom against `x_O` as a scatter plot.

In [ ]:
# your code here

(d) Before plotting, a student keeps only the first Fe-O entry for each formula.

- What information would be lost?

*your answer here*

# Problem 3 (30 pts): the lower convex hull

(a) Print the formula and `x_O` of each Fe-O entry on the hull.

In [ ]:
# your code here

(b) Build the lower hull as two arrays, `hull_x` and `hull_e`, running from pure Fe to pure O.

In [ ]:
# your code here

(c) Remake the plot from Problem 2(c) with the hull drawn as a line over the entries.

In [ ]:
# your code here

(d) A new theoretical Fe-O compound is proposed, with a formation energy below the current hull.

- Explain how adding this entry affects the existing entries' energies above the hull.

*your answer here*

(e) Why might a search for useful materials include entries with positive energy above the hull?

> Look up metastability if it helps.

*your answer here*

# Problem 4 (15 pts): missing values and provenance

Only some crystal entries have a computed bulk modulus.
Run this cell. It reuses `metals` from Problem 1(a).

It queries every elemental entry for those metals, stable or not, and records each bulk modulus.

In [ ]:
moduli = []
with MPRester(API_KEY) as mpr:
    for symbol in metals:
        metal_docs = mpr.materials.summary.search(
            chemsys=symbol,
            num_elements=1,
            fields=['material_id', 'bulk_modulus'],
        )
        for doc in metal_docs:
            try:
                k = doc.bulk_modulus['vrh']   # GPa, Voigt-Reuss-Hill average
            except TypeError:                 # bulk_modulus is None: no elastic calculation
                k = np.nan
            moduli.append({
                'symbol': symbol,
                'material_id': str(doc.material_id),
                'bulk_modulus': k,
            })
metal_moduli = pd.DataFrame(moduli)

(a) Print how many entries in `metal_moduli` have no bulk modulus.

In [ ]:
# your code here

(b) You are selecting entries with a bulk modulus above 100 GPa.

- How should an entry with no bulk modulus appear in your report?

*your answer here*

(c) The `theoretical` field is `True` when Materials Project found no matching experimental structure in the ICSD.

- What does `theoretical` mean for using one of the `fe_o` entries as evidence about real iron oxides?

*your answer here*

(d) Read the v2026.09.28 section of the [Materials Project database changelog](https://docs.materialsproject.org/changes/database-versions).

- Why should a report of hull energies include the database version?

*your answer here*